# Working on the data mattered 20× more than changing the model
*Measured notes from my first Kaggle competition*

This is my first Kaggle competition. When I started, I was comfortable with Python and pandas — I could read data and do basic analysis — but I had never built a model, and I had never actually used AUC.

In one week I went from 0.69 to 0.94 AUC. In this notebook I measured what each change was worth, one at a time. The real thing I learned was not the score — it was how to know whether a change actually works.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e9/train.csv
/kaggle/input/competitions/playground-series-s6e9/test.csv


668,665 rows, 15 columns. The target is Will_Buy_EV: will this person buy an electric vehicle.

The first thing I checked was the target distribution: 17.4% buy, 82.6% don't. This imbalance decides which metric makes sense. If I predict "won't buy" for everyone, I get 82.6% accuracy — that sounds good, but it means I found zero buyers.

That is why the competition uses ROC AUC. AUC asks: if I pick a random buyer and a random non-buyer, does my model give the buyer a higher probability? It measures ranking, not the values themselves.

A small detail I noticed: sample_submission.csv has 0.174645 in every row. That is not a random number — it is the actual positive rate of the training set. I checked, and it matched.

In [2]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/test.csv")

print(train.shape)
print(test.shape)

(668665, 15)
(286571, 14)


In [3]:
test.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level
0,668665,61,67725.0,16.9,2,7,4,4.0,Male,Suburban,Sedan,Yes,No,Low
1,668666,42,152835.0,41.9,2,9,9,4.0,Male,Urban,SUV,No,No,Low
2,668667,68,86877.0,53.3,1,10,11,4.0,Female,Urban,Sedan,No,No,Low
3,668668,39,46794.0,34.1,2,4,8,4.0,Female,Suburban,Sedan,Yes,No,Low
4,668669,55,112172.0,57.2,1,6,4,1.0,Female,Suburban,Sedan,Yes,Yes,Low


In [4]:
sub = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv")
sub.head()

,id,Will_Buy_EV
0,668665,0.174645
1,668666,0.174645
2,668667,0.174645
3,668668,0.174645
4,668669,0.174645


In [5]:
sub.mean()

id             811950.000000
Will_Buy_EV         0.174645
dtype: float64

I made my first submission deliberately useless: the same probability for every row. Score: 0.5.

That is not a failure, it is the zero point. When every prediction is identical there is no ranking — no pair where you can say "this one is more likely." Everything ties, so AUC is 0.5.

It also proved my pipeline worked before I tried to improve anything. I found the format problems at the start instead of after three days of modeling.

In [6]:
sub["Will_Buy_EV"] = 0.5
sub.to_csv('submission.csv', index=False)
sub.shape

(286571, 2)

In [7]:
train.dtypes

id                               int64
Age                              int64
Annual_Income_USD              float64
Daily_Commute_km               float64
Number_of_Cars_Owned             int64
Charging_Stations_Near_Home      int64
Charging_Stations_Near_Work      int64
Environmental_Concern_Level    float64
Gender                          object
City_Type                       object
Current_Car_Type                object
Home_Charging_Possible          object
Subsidy_Available               object
Range_Anxiety_Level             object
Will_Buy_EV                     object
dtype: object

In [8]:
train["Will_Buy_EV"].value_counts()

Will_Buy_EV
No     551886
Yes    116779
Name: count, dtype: int64

In [9]:
y = train["Will_Buy_EV"].map({'No' : 0, 'Yes' : 1})

In [10]:
ozellik = [("Age"),
     ("Annual_Income_USD"),
     ("Daily_Commute_km"),
     ("Number_of_Cars_Owned"),
     ("Charging_Stations_Near_Home"),
     ("Charging_Stations_Near_Work"),
     ("Environmental_Concern_Level")
    ]
X = train[ozellik]

In [11]:
print(X.shape)
print(y.shape)
y.value_counts()

(668665, 7)
(668665,)


Will_Buy_EV
0    551886
1    116779
Name: count, dtype: int64

In [12]:
X_train, X_val, y_train, y_val = train_test_split(X,
                                                  y,
                                                  test_size = 0.2
                                                  ,random_state = 42
                                                  ,stratify = y)

In [13]:
print(X_val.shape)
print(X_train.shape)

(133733, 7)
(534932, 7)


In [14]:
print(y_val.mean())

0.1746464971248682


In [15]:
model = LogisticRegression()

I put every column on the same scale with StandardScaler. AUC 0.8757.

In [16]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
model = LogisticRegression(max_iter= 1000 )
model.fit(X_train_scaled,y_train)
tahmin = model.predict_proba(X_val_scaled)[:, 1]



In [17]:
print(roc_auc_score( y_val , tahmin ))

0.8756990992320652


Honestly, I expected a small improvement. A jump this big surprised me. The model was not bad — it was being fed badly.

In [18]:
X_test = test[ozellik]
print(X_test.shape)

(286571, 7)


In [19]:
X_test_scaled = scaler.transform(X_test)
test_tahmin = model.predict_proba(X_test_scaled)[:, 1]
print(test_tahmin[:5])

[0.1751692  0.57825403 0.19116897 0.10503191 0.01142513]


In [20]:
sub["Will_Buy_EV"] = test_tahmin
sub.to_csv('submission.csv', index=False)
sub.head()
sub.shape

(286571, 2)

Six columns were text and I had not touched them yet. Before encoding I looked at what was inside each one.

Five of them have no order: Gender (Male/Female/Other), City_Type (Urban/Suburban/Rural), Current_Car_Type (Sedan/SUV/Hatchback/Truck), Home_Charging_Possible (Yes/No), Subsidy_Available (Yes/No).

One of them does: Range_Anxiety_Level is Low / Medium / High.

That difference decides the encoding — and the number of categories has nothing to do with it. Gender and City_Type both have 3 values, but they are not the same kind of column.

If I give City_Type the numbers 0, 1, 2, I am telling the model that Suburban is twice Rural, and that Urban is two steps away from Rural but one step from Suburban. None of that is true. So those five get one-hot encoding: each category becomes its own yes/no column, with no size relationship between them.

Range_Anxiety_Level is different. Low < Medium < High is a real order — anxiety actually increases. Mapping it to 0/1/2 keeps information instead of inventing it.

The test I use now: can I sort these values in an order everyone would agree on? If yes, map to numbers. If no, one-hot.

In [21]:
kategorik = [("Gender"),
     ("City_Type"),
     ("Current_Car_Type"),
     ("Home_Charging_Possible"),
     ("Subsidy_Available"),
     ("Range_Anxiety_Level"),
    ]
print(train[kategorik].nunique())

Gender                    3
City_Type                 3
Current_Car_Type          4
Home_Charging_Possible    2
Subsidy_Available         2
Range_Anxiety_Level       3
dtype: int64


In [22]:
for col in kategorik:
    print(col)
    print(train[col].value_counts())

Gender
Gender
Male      367954
Female    295427
Other       5284
Name: count, dtype: int64
City_Type
City_Type
Urban       289305
Suburban    255377
Rural       123983
Name: count, dtype: int64
Current_Car_Type
Current_Car_Type
Sedan        303459
SUV          246545
Hatchback     79438
Truck         39223
Name: count, dtype: int64
Home_Charging_Possible
Home_Charging_Possible
Yes    462677
No     205988
Name: count, dtype: int64
Subsidy_Available
Subsidy_Available
Yes    419909
No     248756
Name: count, dtype: int64
Range_Anxiety_Level
Range_Anxiety_Level
Low       603972
Medium     62499
High        2194
Name: count, dtype: int64


In [23]:
train["Range_Anxiety_Num"] = train["Range_Anxiety_Level"].map({'Low' : 0, 'Medium' : 1, 'High' : 2})
print(train['Range_Anxiety_Num'].value_counts())

Range_Anxiety_Num
0    603972
1     62499
2      2194
Name: count, dtype: int64


In [24]:
kategorik5 = [("Gender"),
     ("City_Type"),
     ("Current_Car_Type"),
     ("Home_Charging_Possible"),
     ("Subsidy_Available")
    ]

In [25]:
tum = ozellik + ['Range_Anxiety_Num'] + kategorik5
X = pd.get_dummies(train[tum], drop_first=True)
print(X.shape)

(668665, 17)


In [26]:
X_train, X_val, y_train, y_val = train_test_split(X,
                                                  y,
                                                  test_size = 0.2
                                                  ,random_state = 42
                                                  ,stratify = y)

In [27]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
model = LogisticRegression(max_iter= 1000 )
model.fit(X_train_scaled,y_train)
tahmin = model.predict_proba(X_val_scaled)[:, 1]

In [28]:
print(roc_auc_score( y_val , tahmin ))

0.9379637067034247


17 features instead of 7. AUC 0.9380.

Until here I used logistic regression, a linear model. For tabular data gradient boosting is usually stronger, so I tried LightGBM.

One thing worth knowing: trees do not need scaling. They split on thresholds — "is income above 50,000?" — and the answer to that question does not change when you rescale the column. Only the ordering matters. So I fed LightGBM the raw features.

In [29]:
model2 = LGBMClassifier(random_state=42)
model2.fit(X_train,y_train)
tahmin2 = model2.predict_proba(X_val)[:, 1]

print(roc_auc_score(y_val, tahmin2))


[LightGBM] [Info] Number of positive: 93423, number of negative: 441509
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015506 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 623
[LightGBM] [Info] Number of data points in the train set: 534932, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174645 -> initscore=-1.553061
[LightGBM] [Info] Start training from score -1.553061
0.9411234081850752


AUC 0.9411. Better, but only by 0.003.

In [30]:
test["Range_Anxiety_Num"] = test["Range_Anxiety_Level"].map({'Low' : 0, 'Medium' : 1, 'High' : 2})
print(test['Range_Anxiety_Num'].value_counts())

Range_Anxiety_Num
0    259247
1     26295
2      1029
Name: count, dtype: int64


In [31]:
X_test = pd.get_dummies(test[tum], drop_first=True)
print(X_test.shape)
print(list(X.columns) == list(X_test.columns))

(286571, 17)
True


In [32]:
tahmin2 = model2.predict_proba(X_test)[:, 1]
sub["Will_Buy_EV"] = tahmin2
sub.to_csv('submission.csv', index=False)
print(sub.head())

       id  Will_Buy_EV
0  668665     0.008772
1  668666     0.017761
2  668667     0.006397
3  668668     0.002982
4  668669     0.019359


## Did my measurement hold up?

This is the part I care about most.

| Model | Local CV | Leaderboard | Gap |
|---|---|---|---|
| LogReg, 7 numeric | 0.8757 | 0.87451 | 0.0012 |
| LightGBM, 17 features | 0.9411 | 0.94082 | 0.0003 |

Twice, my local validation predicted the leaderboard almost exactly.

What made that work:

- `stratify=y` in the split — both halves keep the same 17.4% ratio, so the measurement is not at the mercy of a lucky split
- a fixed `random_state` — if the score changes, the model changed, not the split
- the scaler fitted on the training part only — if it learns the mean from data the model should not see, the measurement is contaminated
- test columns in exactly the same order as training columns — the model reads by position, and a shifted column fails silently, with no error

Why this matters in practice: I get 10 submissions per day. If I had to submit every idea to find out whether it helped, I would run out immediately — and I would slowly overfit to the leaderboard by keeping whatever scored best there.

Now I test locally and only submit the final one or two candidates.

## What actually moved the score

| Change | Type | Gain |
|---|---|---|
| Scaling the features | data | +0.184 |
| Adding the categorical columns | data | +0.062 |
| Switching to LightGBM | model | +0.003 |

Working on the data mattered about 20× more than switching models — and scaling alone mattered 60× more.

I started this competition assuming a better model would be the answer. It was not. Almost everything came from how the data was prepared

## What I would tell someone starting today

Learn pandas properly first. Almost everything that slowed me down was syntax, not concepts. The ML ideas landed quickly — AUC, leakage, why scaling helps. Building a list and assigning it to a variable did not.

My most repeated mistake was writing assignments backwards: putting the thing I wanted to create on the right and the thing I already had on the left.

```python
X_test = scaler.transform(X_test_scaled)   # wrong — X_test_scaled does not exist yet
X_test_scaled = scaler.transform(X_test)   # right
```

But I want to be honest about a tension here. I wish I had known pandas better going in — and at the same time, this week is what made it stick. Practice is what strengthened it.

So my advice is not "finish a course before you start." It is closer to: start, and expect your first week to be mostly syntax. That part passes.